In [2]:
# evaluating fine tuned model on qa test set using accuracy, recall, precision, f-1 score
# side note - we can also try comparing open source llms for our classification task based on this notebook https://colab.research.google.com/github/timothelaborie/text_classification_scripts/blob/main/unsloth_classification.ipynb
# requirements: xrobert model files, test dataset

import os
import io
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from datasets import Dataset
from sklearn.metrics import classification_report

In [3]:
path = os.getcwd() #path should be Evaluatio folder
parent = os.path.join(path, os.pardir) #parent directory
memo_dataset_path = os.path.join(parent, "Data/Memo_Dataset.csv")
df = pd.read_csv(memo_dataset_path)
print(df.head)
df = df.rename(columns={'Question_eng': 'text', 'Trigger': 'label'})
df['label'] = df['label'].astype(int)
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
train_dataset = Dataset.from_pandas(train_df)
train_dataset = train_dataset.remove_columns(["__index_level_0__"])
val_dataset = Dataset.from_pandas(val_df)
val_dataset = val_dataset.remove_columns(["__index_level_0__"])

<bound method NDFrame.head of                                                 Question  \
0      استعملت قطرة لتوسيع حدقة العين وتوسعت، وآلان ل...   
1      أبلغ من العمر ٢٧ سنة تؤلمني رجلي عند ممارسة ال...   
2      السلام عليكمانااسمي عبدالله .وقد قمت بعمل جراح...   
3      قبل عشرة سنوات صابني كهرباء زائدة في الدماغ ور...   
4      السلام عليكم\r\nدكتور منذ شهر تقريبا بدأت اوجا...   
...                                                  ...   
13137  انا متزوج من خمس سنوات ولم يتم الانجاب بل فى و...   
13138  ماذا افعل ابنى عتده ٥ سنوات وينزل دم فى نهاية ...   
13139  هل الميكروب الحلزوني في المعده يؤثر علي صحه ال...   
13140  أنا رجل عمري (59) عاما أعاني من عدم انتصاب الق...   
13141  اريد منكم حلا و اجابتي ان كان هناك حل او علاج ...   

                                            Question_eng  Trigger  
0      I used a drop to dilate the pupil and it dilat...        1  
1      I am 27 years old. My legs hurt when I exercis...        1  
2      Peace be upon you. My name is Abdullah

In [9]:
from transformers import XLMRobertaTokenizer, XLMRobertaForSequenceClassification
import torch

In [10]:
#references: https://huggingface.co/FacebookAI/xlm-roberta-base, https://huggingface.co/docs/transformers/en/model_doc/xlm-roberta#transformers.XLMRobertaForSequenceClassification
#tokenizer and model with random head
tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")
base_model = XLMRobertaForSequenceClassification.from_pretrained("xlm-roberta-base", num_labels=2)

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification #we should try fine tuning XLMRobertaForSequenceClassification as well

tokenizer = AutoTokenizer.from_pretrained("Tanvi0212/xlmr-trigger-detection")
model = AutoModelForSequenceClassification.from_pretrained("Tanvi0212/xlmr-trigger-detection")

In [22]:
text = "Patient had a severe allergic reaction."
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    probs = torch.nn.functional.softmax(logits, dim=-1)
    pred_label_id = torch.argmax(probs, dim=1).item()
    confidence = probs[0][pred_label_id].item()

# check model’s label mapping
print(model.config.id2label)  # should tell you {0: 'Non-trigger', 1: 'Trigger'} or similar

pred_label = model.config.id2label[pred_label_id]

print(f"Predicted label: {pred_label}, Confidence: {confidence:.3f}")


{0: 'LABEL_0', 1: 'LABEL_1'}
Predicted label: LABEL_1, Confidence: 0.960


In [ ]:
# later on can also try comparing performance when using different classification heads

#basic accuracy/precision/recall results for model trained on unbalanced dataset
from transformers import Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report

def preprocess(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

val_tokenized = val_dataset.map(preprocess, batched=True)
val_tokenized = val_tokenized.rename_column("label", "labels")
val_tokenized.set_format("torch")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    return {"accuracy": acc, "f1": f1}

training_args = TrainingArguments(
    output_dir="./eval_results",
    per_device_eval_batch_size=8,
    do_eval=True,
    dataloader_drop_last=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

results = trainer.evaluate()
print("Evaluation metrics:", results)


Map: 100%|██████████| 2629/2629 [00:00<00:00, 10465.89 examples/s]
C:\Users\G34371231\AppData\Local\Temp\ipykernel_38288\585014309.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Evaluation metrics: {'eval_loss': 0.5523592233657837, 'eval_model_preparation_time': 0.0026, 'eval_accuracy': 0.7455306200076075, 'eval_f1': 0.7916536904391155, 'eval_runtime': 16.3845, 'eval_samples_per_second': 160.456, 'eval_steps_per_second': 20.08}


In [7]:
preds = trainer.predict(val_tokenized)
y_true = preds.label_ids
y_pred = np.argmax(preds.predictions, axis=1)

print(classification_report(y_true, y_pred, target_names=["not urgent", "urgent"]))


              precision    recall  f1-score   support

  not urgent       0.72      0.63      0.67      1091
      urgent       0.76      0.83      0.79      1538

    accuracy                           0.75      2629
   macro avg       0.74      0.73      0.73      2629
weighted avg       0.74      0.75      0.74      2629

